**Pontificia Universidad Javeriana**  
**Procesamiento de Alto Volumen de Datos**  
**Fecha:** 3 de febrero 2026  
**Autor:** Sebastian Angulo Vergara  
**Tema:** Introduccion a cuaderno Python con PySpark:  
    - Prueba de servidor Spark en Jupyter  
    - Cluster Spark realizado por mi grupo  
    - Servidor Jupyter como framework o marco trabajo  
    
**Objetivo:** Introduccion al analisis de alto volumen de datos con PYSPARK de Apache y Hadoop HDFS 

In [ ]:
#!pip install findspark

In [1]:
##Importacion de bibliotecas basicas
import os
import sys
import pandas as pd   #---> Para graficar y objetos del dataframe
import numpy as np    #---> Para operaciones matematicas
import matplotlib.pyplot as plt  #---> Para graficar
import seaborn as sns  #---> Para graficar y estadistica

In [2]:
##Importacion de Bibliotecas avanzadas
import findspark
findspark.init('/Almacen/Spark')
from pyspark import SparkContext, SparkConf #---> Para crear el contexto de spark
from pyspark.sql import SparkSession #--->Sesion de Pyspark para consultas SQL

In [3]:
#Se requiere levantar la sesion para trabajar con los servicios
#Basados en Spark: Procesamiento paralelo y distribuido sobre grande volumenes de datoa
configura = SparkConf()
#Se debe asignar un puerto manuamente por la capacidad de los puertos
configura.set("spark.scheduler.mode","FAIR")
configura.set("spark.scheduler","/Almacen/Spark/conf/fairscheduler.xml")
configura.setMaster("spark://10.43.97.166:7077")
configura.setAppName("Stroke_Angulo")
# Se asigna la sesion de spark con su respectiva configuracion
sparkAngulo=SparkSession.builder.config(conf=configura).getOrCreate()
sparkAngulo

## Actividades a Realizar

- **Acceso a HDFS**
- Lectura de Csv: Se empieza con "datos estructurados".
- Cargar en un objeto Dataframe del tipo Spark.
- Presentamos los nombres de las columnas .
- Se cambian los nombres de las columnas.
- Tipo y coherencia de los datos.
- Identificacion de procedimientos para el tratamiento de datos irregulares.
- Estadisticas y graficas exhaustivas.
- Tratar las categorias.

**Acceso a Sistema de Ficheros Distibuidos para Big Data: Hadoop HDFS**

Hadoop HDFS

Hadoop Distributed File System es un sistema de archivos distribuido diseñado para almacenar conjuntos de datos masivos en clústeres de hardware básico con alta tolerancia a fallos. Fragmenta archivos grandes en bloques predeterminados de un tamaño de 128 MB y los replica normalmente 3 veces en distintos nodos, de esta manera garantizando alta disponibilidad, alta tasa de transferencia de datos y escalabilidad. 

    - El fichero se encuentra en el fichero compartido /Almacen.
    - La idea es que el fichero sea leído por todos los nodos al tiempo (Sistema de Ficheros Compartido).

In [ ]:
!$HADOOP_HOME/bin/hadoop fs -ls /csv/stroke_pyspark.csv

In [4]:
## Se crea el dataframe para acceder al Sistema
## de fichero csv como un objeto dataframe PySpark
## EL acceso se hara desde el sistema de ficheros Hadoop HDFS
dfPy00 = sparkAngulo.read.format("csv").option("header","true").load("stroke_pyspark.csv")
dfPy00.show(5)

+-----+------+---+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender|age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+---+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
| 9046|  Male| 67|           0|            1|         Yes|      Private|         Urban|           228.69|36.6|formerly smoked|     1|
|51676|Female| 61|           0|            0|         Yes|Self-employed|         Rural|           202.21| N/A|   never smoked|     1|
|31112|  Male| 80|           0|            1|         Yes|      Private|         Rural|           105.92|32.5|   never smoked|     1|
|60182|Female| 49|           0|            0|         Yes|      Private|         Urban|           171.23|34.4|         smokes|     1|
| 1665|Female| 79|           1|            0|         Yes|Self

**Se hace introduccion sobre los datos de "Ictus" (Stroke) de pacientes almacenados sobre datos estructurados**
#**Diagnostico y tratamiento de datos sobre PySpark**
- Cambio de nombres de columnas  
- Tipos y coherencia de datos  
- Identificacion y tratamiento de datos nulos  
- Estadisticas generales  
- Categorias y cambios sobre el tipo de datos de las categorias  

**1.Cambio de nombres de columnas**

In [5]:
#Imprime la lista con los nombres de las columnas
dfPy00.columns

['id',
 'gender',
 'age',
 'hypertension',
 'heart_disease',
 'ever_married',
 'work_type',
 'Residence_type',
 'avg_glucose_level',
 'bmi',
 'smoking_status',
 'stroke']

In [6]:
## importar funciones
from pyspark.sql import functions as F
##Copio y cambio
NomOriginal = ['id','gender','age','hypertension','heart_disease','ever_married','work_type','Residence_type','avg_glucose_level','bmi','smoking_status','stroke']
NomNueva = ['ID','SEX','EDAD','HIPERT','ENF_CARD','CASADO','TRABAJO','RESIDENCIA','NIV_GLUCOSA','IMC','FUMADOR','ICTUS']

##Version 
## original --> nueva "Atencion que habra un cambio"
dfPy01 = dfPy00
## Se hara un comprimido para el cambio del bucle
for antes, nuevo in zip(dfPy01.columns, NomNueva):
    dfPy01 = dfPy01.withColumnRenamed(antes, nuevo)
    
dfPy01.columns

['ID',
 'SEX',
 'EDAD',
 'HIPERT',
 'ENF_CARD',
 'CASADO',
 'TRABAJO',
 'RESIDENCIA',
 'NIV_GLUCOSA',
 'IMC',
 'FUMADOR',
 'ICTUS']

In [ ]:
dfPy01.show(5)

**Tipo y coherencia de los datos**

In [7]:
###---> Func
dfPy01.printSchema()

root
 |-- ID: string (nullable = true)
 |-- SEX: string (nullable = true)
 |-- EDAD: string (nullable = true)
 |-- HIPERT: string (nullable = true)
 |-- ENF_CARD: string (nullable = true)
 |-- CASADO: string (nullable = true)
 |-- TRABAJO: string (nullable = true)
 |-- RESIDENCIA: string (nullable = true)
 |-- NIV_GLUCOSA: string (nullable = true)
 |-- IMC: string (nullable = true)
 |-- FUMADOR: string (nullable = true)
 |-- ICTUS: string (nullable = true)



In [8]:
## Se cambi el ID por tipo de dato entero
## -> Se cambian todos los enteros

dfPy02 = dfPy01.withColumn("ID",dfPy01.ID.cast("int"))
dfPy02 = dfPy02.withColumn("EDAD",dfPy01.EDAD.cast("int"))
dfPy02 = dfPy02.withColumn("HIPERT",dfPy01.HIPERT.cast("int"))
dfPy02 = dfPy02.withColumn("ENF_CARD",dfPy01.ENF_CARD.cast("int"))
dfPy02 = dfPy02.withColumn("ICTUS",dfPy01.ICTUS.cast("int"))

dfPy02.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- SEX: string (nullable = true)
 |-- EDAD: integer (nullable = true)
 |-- HIPERT: integer (nullable = true)
 |-- ENF_CARD: integer (nullable = true)
 |-- CASADO: string (nullable = true)
 |-- TRABAJO: string (nullable = true)
 |-- RESIDENCIA: string (nullable = true)
 |-- NIV_GLUCOSA: string (nullable = true)
 |-- IMC: string (nullable = true)
 |-- FUMADOR: string (nullable = true)
 |-- ICTUS: integer (nullable = true)



In [9]:
### Se cambian todos los flotantes

dfPy03 = dfPy02.withColumn("NIV_GLUCOSA",dfPy01.NIV_GLUCOSA.cast("double"))
dfPy03 = dfPy03.withColumn("IMC",dfPy01.IMC.cast("double"))

dfPy03.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- SEX: string (nullable = true)
 |-- EDAD: integer (nullable = true)
 |-- HIPERT: integer (nullable = true)
 |-- ENF_CARD: integer (nullable = true)
 |-- CASADO: string (nullable = true)
 |-- TRABAJO: string (nullable = true)
 |-- RESIDENCIA: string (nullable = true)
 |-- NIV_GLUCOSA: double (nullable = true)
 |-- IMC: double (nullable = true)
 |-- FUMADOR: string (nullable = true)
 |-- ICTUS: integer (nullable = true)



Se tratan las Categorias de: SEX|CASADO|TRABAJO|RESIDENCIA|FUMADOR

In [10]:
dfPy03.groupby(["SEX"]).count().show()
dfPy03.groupby(["CASADO"]).count().show()
dfPy03.groupby(["TRABAJO"]).count().show()
dfPy03.groupby(["RESIDENCIA"]).count().show()
dfPy03.groupby(["FUMADOR"]).count().show()

+------+-----+
|   SEX|count|
+------+-----+
|Female| 2994|
| Other|    1|
|  Male| 2115|
+------+-----+

+------+-----+
|CASADO|count|
+------+-----+
|    No| 1757|
|   Yes| 3353|
+------+-----+

+-------------+-----+
|      TRABAJO|count|
+-------------+-----+
| Never_worked|   22|
|Self-employed|  819|
|      Private| 2925|
|     children|  687|
|     Govt_job|  657|
+-------------+-----+

+----------+-----+
|RESIDENCIA|count|
+----------+-----+
|     Urban| 2596|
|     Rural| 2514|
+----------+-----+

+---------------+-----+
|        FUMADOR|count|
+---------------+-----+
|         smokes|  789|
|        Unknown| 1544|
|   never smoked| 1892|
|formerly smoked|  885|
+---------------+-----+



In [13]:
## Se importan las funciones nombradas por F
from pyspark.sql import functions as F
## Cantidad de Datos Nulos o Irregulares
dfPy03.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(),c)).alias(c) for c in dfPy03.columns]).show()

+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+
| ID|SEX|EDAD|HIPERT|ENF_CARD|CASADO|TRABAJO|RESIDENCIA|NIV_GLUCOSA|IMC|FUMADOR|ICTUS|
+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+
|  0|  0|   0|     0|       0|     0|      0|         0|          0|201|      0|    0|
+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+



In [14]:
print(f"Cantidad total de registros {dfPy03.count()}")
print(f"Porcentaje registros nulos {201*100/dfPy03.count()} %")

Cantidad total de registros 5110
Porcentaje registros nulos 3.9334637964774952 %


In [ ]:
** Analisis Preliminar
    -Se observa que la columna IPC tiene aproximadamente el 4% de los datos Nulos
    - La categoria genero un registro otro 
Decisiones
    - Se elimina el registro Other de genero
    - Se saca el promedio de IMC para hacer situacion por los nulos en la columnas.
    - El promedio se toma por genero, por estratos de edades

In [15]:
##Version
dfPy03 = dfPy03


+------+-----+
|   SEX|count|
+------+-----+
|Female| 2994|
| Other|    1|
|  Male| 2115|
+------+-----+



In [59]:
from pyspark.sql import functions as F

# Se cambia el nombre SEX a Genero
dfPy03 = dfPy03.withColumnRenamed('SEX', 'GENERO')
# Se revisa la columna género
dfPy03.groupBy(['GENERO']). count ().show()

+------+-----+
|GENERO|count|
+------+-----+
|Female| 2994|
|  Male| 2115|
+------+-----+



In [60]:
# Se descarta el valor Other
dfpy03 = dfPy03.filter(dfPy03['GENERO'] !="Other")
#Se revisa la columna genero
dfPy03.groupBy(['GENERO']).count().show()

+------+-----+
|GENERO|count|
+------+-----+
|Female| 2994|
|  Male| 2115|
+------+-----+



In [40]:
## Se analiza los datos para contruir los grupos de edad
## Datos minimos y maximos de edad

dfPy03.select(min("EDAD"), max("EDAD")).show()

+---------+---------+
|min(EDAD)|max(EDAD)|
+---------+---------+
|        0|       82|
+---------+---------+



In [54]:
# Se descarta el valor other

dfPy03 = dfPy03.where("GENERO <> 'Other'")

# Se revisa la columna genero: verificacion

dfpy03.groupBy(['GENERO']).count().show()

+------+-----+
|GENERO|count|
+------+-----+
|Female| 2994|
|  Male| 2115|
+------+-----+



In [56]:
# Promedio de IMC por Genero y por edades
# Se toma Los rangos de edades : 0 - 10 || 11 - 20 ||
from pyspark.sql.functions import col, lit, mean

prom10F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") < 10)).select(mean(col ("IMC"))). collect()

In [57]:
print(prom10F)

[Row(avg(IMC)=18.687962962962963)]


In [55]:
## Se imputa los nulos en IMC con el promedio del IMC segun sexo y grupo de edad

# Se obtienen los promedios por sexo y grupo de edad

# Mujeres (Female)
prom10F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") <= 10)).select(mean(col("IMC"))).collect()
prom20F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 10) & (col("EDAD") <= 20)).select(mean(col("IMC"))).collect()
prom30F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 20) & (col("EDAD") <= 30)).select(mean(col("IMC"))).collect()
prom40F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 30) & (col("EDAD") <= 40)).select(mean(col("IMC"))).collect()
prom50F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 40) & (col("EDAD") <= 50)).select(mean(col("IMC"))).collect()
prom60F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 50) & (col("EDAD") <= 60)).select(mean(col("IMC"))).collect()
prom70F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 60) & (col("EDAD") <= 70)).select(mean(col("IMC"))).collect()
prom80F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 70) & (col("EDAD") <= 80)).select(mean(col("IMC"))).collect()
prom90F = dfPy03.where((col("GENERO") == lit("Female")) & (col("EDAD") > 80) & (col("EDAD") <= 90)).select(mean(col("IMC"))).collect()
# Hombres (Male)
prom10M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") <= 10)).select(mean(col("IMC"))).collect()
prom20M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 10) & (col("EDAD") <= 20)).select(mean(col("IMC"))).collect()
prom30M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 20) & (col("EDAD") <= 30)).select(mean(col("IMC"))).collect()
prom40M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 30) & (col("EDAD") <= 40)).select(mean(col("IMC"))).collect()
prom50M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 40) & (col("EDAD") <= 50)).select(mean(col("IMC"))).collect()
prom60M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 50) & (col("EDAD") <= 60)).select(mean(col("IMC"))).collect()
prom70M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 60) & (col("EDAD") <= 70)).select(mean(col("IMC"))).collect()
prom80M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 70) & (col("EDAD") <= 80)).select(mean(col("IMC"))).collect()
prom90M = dfPy03.where((col("GENERO") == lit("Male")) & (col("EDAD") > 80) & (col("EDAD") <= 90)).select(mean(col("IMC"))).collect()

In [58]:
## Se obtienen los promedios
prom10F = __builtins__.round(prom10F[0][0],2)
prom20F = __builtins__.round(prom20F[0][0],2)
prom30F = __builtins__.round(prom30F[0][0],2)
prom40F = __builtins__.round(prom40F[0][0],2)
prom50F = __builtins__.round(prom50F[0][0],2)
prom60F = __builtins__.round(prom60F[0][0],2)
prom70F = __builtins__.round(prom70F[0][0],2)
prom80F = __builtins__.round(prom80F[0][0],2)
prom90F = __builtins__.round(prom90F[0][0],2)

prom10M = __builtins__.round(prom10M[0][0],2)
prom20M = __builtins__.round(prom20M[0][0],2)
prom30M = __builtins__.round(prom30M[0][0],2)
prom40M = __builtins__.round(prom40M[0][0],2)
prom50M = __builtins__.round(prom50M[0][0],2)
prom60M = __builtins__.round(prom60M[0][0],2)
prom70M = __builtins__.round(prom70M[0][0],2)
prom80M = __builtins__.round(prom80M[0][0],2)
prom90M = __builtins__.round(prom90M[0][0],2)

In [ ]:
## Se sustituye Los valores NUlos en La columna IMC por promedios según Genero Female
dfPy04 = dfPy03. withColum ("IMC", F.when( (dfPy03 ["GENERO"] == 'Female') & (dfPy03["IMC"]. isNu11() & (dfPy03["EDAD"] < 10)),
round (prom10F[0][0],2)). otherwise(dfPy03[ "IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when((dfPy84|"GENER0"] == "Female") & (dfPy04|"MC"]. isNu11() & (dfPy84["EDAD"] < 20)),
round (prom20F(@] [0],2)). otherwise(dfPy04[ "IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when((dfPy04["GENERO"] == 'Female") & (dfPy04["IMC"].isNu11() & (dfPy04["EDAD"] < 30)),
round (prom30F[®][0],2)). otherwise(dfPy04 ["IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when((dfPy04|"GENERO"] == "Female") & (dfPy04|"IMC"].isNull() & (dfPy04["EDAD"] < 40)), round (prom40F[0] [0],2)). otherwise(dfPy04("IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when ((dfPy84|"GENERO"] == 'Female") & (dfPy®4["IMC"] .isNu11() & (dfPy04[ "EDAD"] < 50)),
round (prom50F[®][0],2)). otherwise(dfPy04["IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when((dfPy84[ "GENERO"] == 'Female") & (dfPy04["IMC"]. isNu11() & (dfPy04[ "EDAD"] < 60)),
round (prom6@F[®][0],2)) . otherwise(dfPy04["MC"]))
dfPy04 = dfPy04.withColum(" IMC", F.when( (dfPy84[ "GENERO"] == 'Female") & (dfPy04["IMC"]. isNu11() & (dfPy04["EDAD"] < 70)),
round (prom70F[0][0],2)) .otherwise(dfPy04["IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when((dfPy04[ "GENERO") == "Female') & (dfPy04["IMC"] . isNu11() & (dfPy84["EDAD"] < 80)),
round (prom8@F[0][®],2)) .otherwise(dfPy04[ "IMC"]))
dfPy04 = dfPy04.withColumn("IMC", F.when ((dfPy84("GENERO"] == "Female") & (dfPy04["MC"] . isNu11() & (dfPy84|"EDAD"] < 90)),
round (prom90F[0][0],2)). otherwise(dfPy04["IMC"]))